# 12. Sensitivity analysis

## Numerical experiments - Week 45/2025

_Boyan Mihaylov, MSc Computational Science (UVA/VU)_

The procedure presented below performs Bayesian Model Averaging (BMA) on the broad collection of models to
- assess their likelihood of representing the data (with its inherent variance),
- penalize overly complex models,
- assess the models' predictive capabilities
- and explore the combined predictive power of weighted combinations of models.

The aim is to obtain a list of models ranked by their pseudo-BMA weights, which evaluate predictive accuracy under uncertainty and are thus a better criterion than RMSE of model fits to data.

## Prerequisite libraries

In [ ]:
using PyPlot
using Revise

# include("../src/conversions.jl")
# include("../src/diffusion.jl")
# include("../src/setup.jl")
# include("../src/plotting.jl")
# include("../src/analysis.jl")
# include("../src/datautils.jl")
# include("../src/germstats.jl")

Revise.includet("../src/conversions.jl")
Revise.includet("../src/diffusion.jl")
Revise.includet("../src/setup.jl")
Revise.includet("../src/plotting.jl")
Revise.includet("../src/analysis.jl")
Revise.includet("../src/datautils.jl")
Revise.includet("../src/germstats.jl")

using .Conversions
using .Diffusion
using .Setup
using .Plotting
using .Analysis
using .DataUtils
using .GermStats

## 1. Parameter priors

The first step is to assume prior distributions of the model parameters, which encode the expectations of plausible parameter values before relating them to data.

### 1.1. Parameter summary

The free parameters in the germination models are:

- $P_s^\textrm{I}$ - the permeation coefficient of the inhibitor molecule through the cell wall;
- $P_s^\textrm{C}$ - the permeation coefficient of the inducer molecule (carbon source) through the cell wall;
- $K_\textrm{I}$ - the half-saturation constant of the inhibitor when directly blocking the inducing signal;
- $K_T^\textrm{I}$ - the half-saturation constant of the inhibitor when shifting the induction threshold;
- $K_T^\textrm{C}$ - the half-saturation constant of the inducer when shifting the inhibition threshold;
- $s$ - scaling factor of the permeability perturbation caused by the inducing signal;
- $b$ - scaling factor of the permeability perturbation caused by the inhibitory signal;
- $k_\textrm{I}$ - scaling factor of the induction threshold shift caused by the inhibitory signal;
- $k_\textrm{C}$ - scaling factor of the inhibition threshold shift caused by the inducing signal;
- $n$ - Hill exponent of the direct inhibition of the inducing signal;
- $\mu_\gamma$ - mean of the inhibition threshold;
- $\sigma_\gamma$ - standard deviation of the inhbition threshold;
- $\mu_\omega$ - mean of the induction threshold;
- $\sigma_\omega$ - standard deviation of the induction threshold;
- $\mu_\psi$ - mean of the initial inhibitor concentration in the spore;
- $\sigma_\psi$ - standard deviation of the initial inhibitor concentration in the spore.

Some of these parameters are independent of the type of carbon source, others are inducer-specific and therefore get a specific instance (with its own distribution) in each inducer case. The inducer-specific parameters are $P_s^\textrm{C}$, $K_\textrm{I}$, $K_T^\textrm{I}$, $K_T^\textrm{C}$, $s$, $k_\textrm{I}$, $k_\textrm{C}$, $n$, $\mu_\omega$, $\sigma_\omega$. On the other hand, $P_s^\textrm{I}$, $b$, $\mu_\gamma$, $\sigma_\gamma$, $\mu_\psi$ and $\sigma_\psi$ are general for all inducer cases.

### 1.2. Determining the priors

Since an exact quantification of the prior distributions is generally difficult to infer from literature, several approaches can be employed to narrow down the bounds, shape an scale of each parameter prior:

1. Use biologically plausible bounds wherever known.